# 01 — Preflight and train the Transformer control
Freezes a shared T4-safe experiment contract, trains one ordered pass over 16M bases, and evaluates both within-genome and ANI-held-out generalization.

In [ ]:
import importlib
from pathlib import Path
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')
ATCG_COMMIT = '59cda3673f0cf9ab2720ea8d47fd47605ecaa865'
!test -d /content/ATCG-FM || git clone https://github.com/DRAGGON-Lab/ATCG-FM.git /content/ATCG-FM
!git -C /content/ATCG-FM fetch origin {ATCG_COMMIT}
!git -C /content/ATCG-FM checkout --detach {ATCG_COMMIT}
repo = Path('/content/ATCG-FM')
packages = [repo / 'packages/atcg-sequence', repo / 'packages/atcg-models', repo / 'packages/atcg-runtime']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps', *map(str, packages)])
for package in reversed(packages):
    sys.path.insert(0, str(package / 'src'))
importlib.invalidate_caches()
from atcg.models import GenomicLanguageModel as _ATCGModelCheck
from atcg.runtime import OrderedComparisonConfig as _ATCGRuntimeCheck
from atcg.sequence import FixedAlphabetTokenizer as _ATCGSequenceCheck
print('ATCG imports ready:', _ATCGModelCheck.__module__, _ATCGRuntimeCheck.__module__, _ATCGSequenceCheck.__module__)


In [ ]:
from pathlib import Path
from dataclasses import asdict
import gzip, hashlib, json, shutil, time, torch
from atcg.models import GenomicLanguageModel, attention_tiny, titans_memory_tiny
from atcg.runtime import CudaUtilizationSampler, OrderedComparisonConfig, OrderedComparisonTrainer, ProfileMeasurement, benchmark_ordered_candidate, ordered_dataset_fingerprint, restore_ordered_trainer, save_checkpoint, select_largest_fitting_profile, validate_ordered_model, validate_repeat_contexts
from atcg.sequence import FixedAlphabetTokenizer, OrderedCausalStreamDataset, parse_fasta

if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime in Colab')
gpu = torch.cuda.get_device_properties(0)
if 'T4' not in gpu.name: raise RuntimeError(f'This protocol is calibrated for a T4, found {gpu.name}')
ROOT = Path('/content/drive/MyDrive/ATCG-FM/ecoli_hybrid_v1'); STAGE_NAME = 'stage-small'; MAX_TRAIN_HOURS = 2.5; STAGE = ROOT / STAGE_NAME
SPLITS = ('train','validation_within','test_within','validation_clade','test_clade'); EVALUATION_SPLITS = SPLITS[1:]
LOCAL = Path('/content/ecoli_hybrid_stage_small'); LOCAL.mkdir(exist_ok=True)
for name in (*[f'{split}.fa.gz' for split in SPLITS], 'manifest.json'): shutil.copy2(STAGE / name, LOCAL / name)
manifest = json.loads((LOCAL / 'manifest.json').read_text())
if manifest['schema_version'] != 2: raise RuntimeError('hybrid manifest schema mismatch')
for split in SPLITS:
    digest = hashlib.sha256((LOCAL / f'{split}.fa.gz').read_bytes()).hexdigest()
    if digest != manifest['split_sha256'][split]: raise RuntimeError(f'{split} checksum mismatch')
def records(name):
    with gzip.open(LOCAL / f'{name}.fa.gz', 'rt') as handle: return parse_fasta(handle, source=str(LOCAL / f'{name}.fa.gz'))
def dataset(values): return OrderedCausalStreamDataset(values, tokenizer, segment_length=128, gradient_horizon=1, include_bos=True, include_eos=False)
tokenizer = FixedAlphabetTokenizer(); train_records = records('train'); train_data = dataset(train_records)
train_tokens = sum(len(segment.target_ids) for horizon in train_data for segment in horizon.segments)
schedule_hash = ordered_dataset_fingerprint(train_data)
print(gpu.name, round(gpu.total_memory / 1024**3, 2), 'GiB', len(train_records), train_tokens, schedule_hash)


In [ ]:
profiles = ({'id':'d64-l2','d_model':64,'layers':2,'heads':4}, {'id':'d96-l3','d_model':96,'layers':3,'heads':4}, {'id':'d128-l4','d_model':128,'layers':4,'heads':4}, {'id':'d192-l4','d_model':192,'layers':4,'heads':4})
base = OrderedComparisonConfig(global_batch_size=32, microbatch_size=1, precision='float16', device='cuda', seed=17)
profile_rows = []; titan_benchmarks = {}
for candidate_profile in profiles:
    def factory(p=candidate_profile): return GenomicLanguageModel(titans_memory_tiny(tokenizer.vocab_size, max_seq_len=128, d_model=p['d_model'], n_layers=p['layers'], expansion_factor=2, projection_kernel_size=4))
    measured = benchmark_ordered_candidate(factory, train_data, pad_id=tokenizer.pad_id, base_config=base, microbatch_sizes=(1,2,4,8,16,32), timed_global_batches=1)
    safe = [row for row in measured if row.peak_memory_bytes <= 13 * 1024**3]
    if not safe: continue
    best = max(safe, key=lambda row: row.tokens_per_second); titan_benchmarks[candidate_profile['id']] = asdict(best)
    profile_rows.append(ProfileMeasurement(candidate_profile['id'], train_tokens / best.tokens_per_second, best.peak_memory_bytes, candidate_profile['d_model'], candidate_profile['layers']))
selected = select_largest_fitting_profile(profile_rows, maximum_seconds=MAX_TRAIN_HOURS * 3600, maximum_memory_bytes=13 * 1024**3)
profile = next(value for value in profiles if value['id'] == selected.profile_id)
def attention_factory(): return GenomicLanguageModel(attention_tiny(tokenizer.vocab_size, max_seq_len=128, d_model=profile['d_model'], n_layers=profile['layers'], n_heads=profile['heads']))
attention_measured = benchmark_ordered_candidate(attention_factory, train_data, pad_id=tokenizer.pad_id, base_config=base, microbatch_sizes=(1,2,4,8,16,32), timed_global_batches=1)
attention_best = max((row for row in attention_measured if row.peak_memory_bytes <= 13 * 1024**3), key=lambda row: row.tokens_per_second)
experiment = {'schema_version':2,'comparison_id':'ecoli-hybrid-stage-small-mixer-v1','stage':STAGE_NAME,'maximum_train_hours':MAX_TRAIN_HOURS,'dataset_fingerprint':manifest['dataset_fingerprint'],'split_sha256':manifest['split_sha256'],'evaluation_splits':list(EVALUATION_SPLITS),'schedule_hash':schedule_hash,'train_tokens':train_tokens,'segment_length':128,'stream_length':manifest['stream_length'],'gradient_horizon':1,'global_batch_size':32,'seed':17,'learning_rate':3e-4,'weight_decay':0.1,'precision':'float16','epochs':1,'profile':profile,'attention_microbatch':attention_best.microbatch_size,'titans_microbatch':titan_benchmarks[profile['id']]['microbatch_size'],'preflight':{'profiles':[asdict(row) for row in profile_rows],'attention':asdict(attention_best),'titans':titan_benchmarks[profile['id']],'gpu':gpu.name}}
(STAGE / 'experiment_config.json').write_text(json.dumps(experiment, indent=2, sort_keys=True) + '\n')
print(json.dumps(experiment, indent=2))


In [ ]:
run_dir = ROOT / 'runs' / STAGE_NAME / 'attention-seed17'; run_dir.mkdir(parents=True, exist_ok=True)
torch.manual_seed(experiment['seed']); model = attention_factory()
config = OrderedComparisonConfig(global_batch_size=experiment['global_batch_size'], microbatch_size=experiment['attention_microbatch'], learning_rate=experiment['learning_rate'], weight_decay=experiment['weight_decay'], precision='float16', device='cuda', seed=experiment['seed'])
trainer = OrderedComparisonTrainer(model, pad_id=tokenizer.pad_id, segment_length=128, config=config)
resume_path = run_dir / 'resume.pt'; metrics_path = run_dir / 'metrics.jsonl'
metrics = [json.loads(line) for line in metrics_path.read_text().splitlines()] if resume_path.exists() and metrics_path.exists() else []
start_batch = restore_ordered_trainer(str(resume_path), trainer, dataset_fingerprint=experiment['dataset_fingerprint']) if resume_path.exists() else 0
torch.cuda.reset_peak_memory_stats()
with CudaUtilizationSampler() as gpu_sampler:
    for batch_index, horizons in enumerate(train_data.iter_batches(config.global_batch_size)):
        if batch_index < start_batch: continue
        row = trainer.train_global_batch(horizons); metrics.append(asdict(row))
        if (batch_index + 1) % 100 == 0:
            save_checkpoint(resume_path, model=model, optimizer=trainer.optimizer, training_state=trainer.state, stream_state=trainer.state_store.state_dict(), grad_scaler_state=trainer.scaler.state_dict(), experiment_state={'dataset_fingerprint':experiment['dataset_fingerprint'],'schedule_hash':schedule_hash,'global_batch_index':batch_index+1})
            with metrics_path.open('w') as handle:
                for value in metrics: handle.write(json.dumps(value, sort_keys=True) + '\n')
        if batch_index % 100 == 0: print(batch_index, metrics[-1])
wall_time = sum(value['elapsed_seconds'] for value in metrics); training_peak = torch.cuda.max_memory_allocated()
checkpoint = save_checkpoint(run_dir / 'last.pt', model=model, optimizer=trainer.optimizer, training_state=trainer.state, stream_state=trainer.state_store.state_dict(), grad_scaler_state=trainer.scaler.state_dict(), experiment_state={'dataset_fingerprint':experiment['dataset_fingerprint'],'schedule_hash':schedule_hash,'global_batch_index':len(metrics)})
with metrics_path.open('w') as handle:
    for value in metrics: handle.write(json.dumps(value, sort_keys=True) + '\n')
resume_path.unlink(missing_ok=True)
if trainer.state.tokens_seen != train_tokens: raise RuntimeError('training did not consume exactly one shared token pass')


In [ ]:
evaluation_records = {name:records(name) for name in EVALUATION_SPLITS}
evaluation = {name:asdict(validate_ordered_model(model, dataset(values), pad_id=tokenizer.pad_id, batch_size=32, device='cuda', offset_boundaries=(128,1024,4096,16384))) for name,values in evaluation_records.items()}
def stream_scores(name):
    output = []
    for record in evaluation_records[name]:
        score = validate_ordered_model(model, dataset([record]), pad_id=tokenizer.pad_id, batch_size=1, device='cuda')
        output.append({'stream_id':record.identifier,'accession':record.identifier.split('__',1)[0],'bits_per_token':score.bits_per_token,'token_count':score.token_count})
    return output
stream_evaluation = {name:stream_scores(name) for name in ('test_within','test_clade')}
repeat_context = asdict(validate_repeat_contexts(model, dataset(evaluation_records['test_within']), source_records=evaluation_records['test_within'], training_records=train_records, device='cuda', max_tokens=131_072))
result = {'candidate_id':'attention','experiment':experiment,'parameters':model.parameter_count(),'recurrent_state_elements_per_stream':model.recurrent_state_elements(),'training':{'steps':trainer.state.step,'tokens':trainer.state.tokens_seen,'wall_time_seconds':wall_time,'tokens_per_second':trainer.state.tokens_seen/wall_time,'peak_memory_bytes':training_peak,'gpu_utilization':gpu_sampler.summary()},'evaluation':evaluation,'stream_evaluation':stream_evaluation,'repeat_context':repeat_context,'checkpoint':str(checkpoint)}
(run_dir / 'result.json').write_text(json.dumps(result, indent=2, sort_keys=True) + '\n')
print(json.dumps(result, indent=2))
